In [1]:
from dotenv import load_dotenv
from agents import Runner, Agent, trace
from pypdf import PdfReader
from pydantic import BaseModel
import gradio as gr
import asyncio

c:\Users\Sayak\Desktop\Projects\Student's Corner\python-backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv(override=True)

True

In [3]:
user_resume_render = PdfReader("user/Sayak_Resume.pdf")
user_resume = ""
for page in user_resume_render.pages:
    text = page.extract_text()
    if text:
        user_resume += text
print(user_resume)

Sayak Mitra Majumder
♂¶ap-¶arker-altPune /envel⌢pesayakmitra16@gmail.com ♂phone-alt6370275513
/linkedin-inSayak Mitra Majumder /githubReversibleWizard
Summary
Computer Science student with project experience in backend development using Python, Flask, and SQL. Worked on AI-powered
applications and data-driven projects, with strong interest in Intelligent Automation and emerging AI technologies. Eager to
contribute to scalable solutions and grow as a developer.
Education
Kalinga Institute of Industrial T echnology
B.Tech in Computer Science and Communication Engineering
Sept 2022 – Present
◦ CGP A:8.55/10.0
Internship
Keploy API F ellowship
Backend Fellow – Open Source & API Testing
June 2025 – July 2025
◦ Completed a series of hands-on assignments focused on API testing, mocking, and traffic replay using Keploy.
◦ Designed and tested APIs using Postman and integrated Keploy into the request-response workflow.
◦ Gained practical experience and knowledge in software testing and CI/CD pra

In [4]:
print("h\ne\nl\nl")

h
e
l
l


In [27]:
class AnswerReview(BaseModel):
    question: str
    user_answer: str
    user_answer_review: str
    next_question: str
    completed: bool

name = "Sayak"
time = "15 min"
user_experience = "Student"
work_experience = "0 years"
current_confidence_level = "Low"
interviewer_instruction = f"""
You are a Interviewer for a company XYZ. 
You are taking interview of {name} and you are give his resume data down below.
The user is a {user_experience} with a prior experience of {work_experience}
Read the resume data properly and understand the user's work experience[if available], education, skills and projects.
Your task is ask the user question and help the user for their interview.
Remeber the user is {current_confidence_level} in confidence so ask him appropriately.
Ask a single question at a time. After the user responds ask him the appropriate follow-up question.
Remember the user has specified the interview duration as {time} so you should decide the number of question accordingly.
Always start by asking the user their introduction.
Keep the questions short and crisp.
For every user answer you must:

1. Review the user's answer.
2. Provide constructive feedback.
3. Ask the next interview question.

The interview begins with the question and return this qestion whenever you get a message as "begin":
"Introduce yourself".

Review the user's response and write your review/ assesment about the answer in user_answer_review.
\n \n
"""
interviewer_instruction += f"Resume data: \n \n {user_resume}"


In [6]:
interviewer = Agent(
    name="Interviewer", 
    instructions=interviewer_instruction, 
    output_type=AnswerReview, 
    model="gpt-5-mini-2025-08-07"
)

In [7]:
user_response = "Hello, I'm Sayak Mitra Majumder, a final-year B.Tech student in Computer Science and Communication Engineering at KIIT University, with a CGPA of 8.55. \
I have experience in Python, Flask, and Django, along with some exposure to full-stack development. \
I've worked on projects such as a Movie Recommender System and an Intelligent Tutoring System, and I also have hands-on experience with cloud technologies, particularly AWS. \
Recently, I've developed a strong interest in Large Language Models and Agentic AI systems, and I'm eager to apply my technical skills to solving real-world problems and contributing to impactful projects."
with trace("InterviewTrial"):
    result = await Runner.run(interviewer, user_response)

print(result)

RunResult:
- Last agent: Agent(name="Interviewer", ...)
- Final output (AnswerReview):
    {
      "question": "Introduce yourself",
      "user_answer": "Hello, I'm Sayak Mitra Majumder, a final-year B.Tech student in Computer Science and Communication Engineering at KIIT University, with a CGPA of 8.55. I have experience in Python, Flask, and Django, along with some exposure to full-stack development. I've worked on projects such as a Movie Recommender System and an Intelligent Tutoring System, and I also have hands-on experience with cloud technologies, particularly AWS. Recently, I've developed a strong interest in Large Language Models and Agentic AI systems, and I'm eager to apply my technical skills to solving real-world problems and contributing to impactful projects.",
      "user_answer_review": "Assessment:\n- Strengths: Clear and concise. You communicated your education, core technical skills (Python, Flask, Django), relevant projects, cloud exposure, and current interest i

In [8]:
print(result.final_output.user_answer_review)

Assessment:
- Strengths: Clear and concise. You communicated your education, core technical skills (Python, Flask, Django), relevant projects, cloud exposure, and current interest in LLMs/Agentic AI. Mentioning CGPA and specific projects gives credibility.
- Improvements: Add one or two concrete specifics to make the intro more memorable — for example, your role in a project (what you built), a measurable outcome (e.g., reduced latency, accuracy, or number of users), or the tech you used in one sentence. This helps interviewers quickly understand your impact and depth.
- Tone and confidence: Good professional tone. To boost perceived confidence, phrase achievements actively ("I built X that does Y") rather than listing skills.
Suggested concise revision you can try next time:
"I’m Sayak, a final-year CSE student (CGPA 8.55). I built a Flask-based Movie Recommender that uses TF-IDF and cosine similarity to personalize suggestions and stores user history in MongoDB. I’ve completed backen

In [9]:
# Version 1.0 -> Chat
class InterviewContext(BaseModel):
    history: list[AnswerReview] = []
    current_question: str = "Introduce yourself"

context = InterviewContext()

def chat(message, history):

    question = context.current_question

    async def run_agent():
        result = await Runner.run(
            interviewer,
            f"""
            Question asked:
            {question}

            User answer:
            {message}
            """,
            context=context
        )
        return result

    result = asyncio.run(run_agent())
    print(result)

    review = result.final_output

    # store history
    context.history.append(review)

    # update next question
    context.current_question = review.next_question

    response = f"""
Review:
{review.user_answer_review}

Next Question:
{review.next_question}
"""

    return review.next_question

In [18]:
demo = gr.ChatInterface(
    fn=chat,
    title="AI Interviewer",
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [23]:
for history in context.history:    
    print(history)

question='Introduce yourself' user_answer="Hello, I'm Sayak Mitra Majumder, a final-year B.Tech student in Computer Science and Communication Engineering at KIIT University, with a CGPA of 8.55. \\I have experience in Python, Flask, and Django, along with some exposure to full-stack development. \\I've worked on projects such as a Movie Recommender System and an Intelligent Tutoring System, and I also have hands-on experience with cloud technologies, particularly AWS. \\Recently, I've developed a strong interest in Large Language Models and Agentic AI systems, and I'm eager to apply my technical skills to solving real-world problems and contributing to impactful projects." user_answer_review='Good points:\n- Clear and concise introduction with education and CGPA.\n- You named relevant technologies (Python, Flask, Django) and projects, which shows practical experience.\n- Mentioning AWS and interest in LLMs/Agentic AI signals alignment with current industry trends.\n\nAreas to improve:\

In [ ]:
# Version 1.1 -> Interface changes
context = InterviewContext()

def chat(message, history):

    question = context.current_question

    async def run_agent():
        result = await Runner.run(
            interviewer,
            f"""
                Question asked: {question}

                User answer: {message}
            """,
            context=context
        )
        return result

    result = asyncio.run(run_agent())

    review = result.final_output
    print(review)

    # store interview history
    context.history.append(review)

    # update next question
    context.current_question = review.next_question

    response = f"""
        Answer Review:
        {review.user_answer_review}

        Next Question:
        {review.next_question}
        """
    next_question = review.next_question

    return next_question


def load_first_question():
    # Older Gradio uses (user_msg, bot_msg) tuples — None means no user message
    return [{"role": "assistant", "content": context.current_question}]

SyntaxError: parameter without a default follows parameter with a default (164547230.py, line 4)

In [29]:
with gr.Blocks() as demo:
    chatbot = gr.Chatbot()  # No type argument
    interface = gr.ChatInterface(fn=chat, chatbot=chatbot)
    # if len(context.history) == 0:
    #     demo.load(fn=load_first_question, inputs=None, outputs=chatbot)

demo.launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


In [25]:
print(len(context.history))

0


In [30]:
def get_first_message():
    # Agent generates the first message — no hardcoding
    result = asyncio.run(Runner.run(interviewer, "begin", context=context))
    first_msg = result.final_output.next_question
    context.current_question = first_msg
    return [{"role": "assistant", "content": first_msg}]

def chat(message, history):
    result = asyncio.run(Runner.run(
        interviewer,
        f"Question asked: {context.current_question}\nUser answer: {message}",
        context=context
    ))
    review = result.final_output
    context.history.append(review)
    context.current_question = review.next_question
    return review.next_question

with gr.Blocks() as demo:
    chatbot = gr.Chatbot()
    interface = gr.ChatInterface(fn=chat, chatbot=chatbot)
    demo.load(fn=get_first_message, inputs=None, outputs=chatbot)

demo.launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.
